# Analysis Notebook — Lab Assignment 02
## Failure Modes in Chained Tool-Calling Agents
### MRA2026004 | IIIT Allahabad

This notebook loads the simulation results and reproduces the key figures and statistics reported in the paper.

In [ ]:
import csv
import math

# Load results
results = []
with open('../results/trajectory_metrics.csv', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        results.append({
            'chain_length': int(row['chain_length']),
            'mode': row['mode'],
            'task_success_rate': float(row['task_success_rate']),
            'divergence_rate': float(row['divergence_rate']),
            'avg_irreversible_errors': float(row['avg_irreversible_errors']),
            'avg_human_interventions': float(row['avg_human_interventions'])
        })

print(f'Loaded {len(results)} result rows')
print('Columns:', list(results[0].keys()))

In [ ]:
# --- Table IV Reproduction ---
print(f'{'n':>6} | {'Mode':>10} | {'Success':>10} | {'Divergence':>12} | {'Irrev. Err':>12} | {'Interventions':>14}')
print('-' * 72)
for r in results:
    print(f"{r['chain_length']:>6} | {r['mode']:>10} | {r['task_success_rate']*100:>9.1f}% | {r['divergence_rate']*100:>11.1f}% | {r['avg_irreversible_errors']:>12.3f} | {r['avg_human_interventions']:>14.2f}")

In [ ]:
# --- Key Metric: Improvement of Adaptive over Unsupervised ---
none_data = {r['chain_length']: r for r in results if r['mode'] == 'none'}
adaptive_data = {r['chain_length']: r for r in results if r['mode'] == 'adaptive'}

print('Adaptive vs Unsupervised — Irreversible Error Reduction:')
print(f'{'n':>6} | {'None Err':>10} | {'Adapt Err':>10} | {'Reduction':>12}')
print('-' * 48)
for n in sorted(none_data.keys()):
    none_err = none_data[n]['avg_irreversible_errors']
    adapt_err = adaptive_data[n]['avg_irreversible_errors']
    reduction = (1 - adapt_err / none_err) * 100 if none_err > 0 else 0
    print(f'{n:>6} | {none_err:>10.3f} | {adapt_err:>10.3f} | {reduction:>11.1f}%')

In [ ]:
# --- Theoretical vs Empirical comparison at n=50 ---
p_base = 0.95
n = 50
theoretical_independent = p_base ** n
empirical_unsupervised = 0.085  # from results

print(f'Theoretical (independent, p=0.95, n=50): {theoretical_independent*100:.2f}%')
print(f'Empirical unsupervised (autocorrelated): {empirical_unsupervised*100:.1f}%')
print(f'WebArena GPT-4 empirical:                14.41%')
print(f'Adaptive checkpointing (proposed):        79.0%')
print()
print(f'=> Autocorrelation degrades success by {(theoretical_independent - empirical_unsupervised)*100:.1f} percentage points beyond i.i.d. floor')

In [ ]:
# --- ASCII Trajectory Collapse Chart (no matplotlib required) ---
none_success = {r['chain_length']: r['task_success_rate'] for r in results if r['mode'] == 'none'}
fixed_success = {r['chain_length']: r['task_success_rate'] for r in results if r['mode'] == 'fixed_5'}
adaptive_success = {r['chain_length']: r['task_success_rate'] for r in results if r['mode'] == 'adaptive'}

print('Task Success Rate vs Chain Length')
print(f'{'n':>4} | {'None':>8} | {'Fixed-5':>8} | {'Adaptive':>10}')
print('-' * 40)
for n in sorted(none_success.keys()):
    bar_none = '#' * int(none_success[n] * 20)
    print(f'{n:>4} | {none_success[n]*100:>7.1f}% | {fixed_success[n]*100:>7.1f}% | {adaptive_success[n]*100:>9.1f}%  {bar_none}')